# Dana's Bakery: LangGraph Order Workflow

A deterministic workflow based on Dana's bakery rules. Morning inventory is fixed for every independent order; this example does not reserve or decrement stock between test runs.

## 1. Setup

Run this cell in a fresh Colab runtime. No model, API key, or external service is used.

In [ ]:
%pip install -q langgraph

## 2. Policy and typed state

Inputs are normalized at validation. Unknown products, invalid quantities, and malformed weekdays are rejected. A substitute must cover the full requested quantity.

In [ ]:
from typing import Literal, TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

PRODUCTS = {
    "challah": {"price": 18, "stock": 10},
    "babka": {"price": 45, "stock": 0},
    "rugelach": {"price": 30, "stock": 3},
    "sourdough": {"price": 25, "stock": 4},
}
SUCCESSORS = {
    "babka": ["rugelach", "sourdough", "challah"],
    "rugelach": ["sourdough", "challah"],
    "sourdough": ["challah"],
    "challah": [],
}
WEEKDAYS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}
PUBLIC_KEYS = {"status", "item", "delivery", "total", "message", "steps"}

class OrderState(TypedDict, total=False):
    item: str
    quantity: int
    day: str
    city: str
    status: str
    delivery: str
    total: float
    message: str
    steps: list[str]
    requested_item: str
    selected_item: str
    candidate_index: int
    suggestions_tried: int
    requested_in_stock: bool
    candidate_available: bool
    substitute_in_stock: bool
    goods_subtotal: float
    delivery_fee: float
    discount: float
    invalid_reason: str
    fulfillment_note: str


## 3. Graph nodes and routing

Substitutes are checked in order. Each candidate consumes one of the two suggestion attempts even when stock is insufficient. The first substitute with sufficient stock is selected.

In [ ]:
def add_step(state: OrderState, entry: str) -> list[str]:
    return [*state.get("steps", []), entry]

def validate_order(state: OrderState) -> dict:
    item = state.get("item")
    day = state.get("day")
    city = state.get("city", "")
    quantity = state.get("quantity")
    normalized_item = item.strip().casefold() if isinstance(item, str) else ""
    normalized_day = day.strip().casefold() if isinstance(day, str) else ""
    normalized_city = city.strip() if isinstance(city, str) else ""
    reason = ""
    if normalized_item not in PRODUCTS:
        reason = "I couldn't understand that product. Please choose challah, babka, rugelach, or sourdough."
    elif isinstance(quantity, bool) or not isinstance(quantity, int) or quantity <= 0:
        reason = "I couldn't understand the quantity. Please send a positive whole number."
    elif normalized_day not in WEEKDAYS:
        reason = "I couldn't understand the day. Please send a weekday name."
    return {
        "requested_item": normalized_item, "selected_item": normalized_item,
        "day": normalized_day.title(), "city": normalized_city,
        "status": "rejected" if reason else "", "delivery": "pickup", "total": 0.0,
        "message": "", "candidate_index": 0, "suggestions_tried": 0,
        "goods_subtotal": 0.0, "delivery_fee": 0.0, "discount": 0.0,
        "invalid_reason": reason,
        "steps": add_step(state, "Validated and normalized the order." if not reason else "Validated the order and found invalid input."),
    }

def route_after_validation(state: OrderState) -> Literal["reject_order", "check_requested_stock"]:
    return "reject_order" if state.get("invalid_reason") else "check_requested_stock"

def check_requested_stock(state: OrderState) -> dict:
    item = state["requested_item"]
    enough = PRODUCTS[item]["stock"] >= state["quantity"]
    return {"requested_in_stock": enough,
            "steps": add_step(state, f"Checked requested item {item}: {'enough stock' if enough else 'not enough stock'}.")}

def route_after_requested_stock(state: OrderState) -> Literal["prepare_fulfillment", "choose_substitute"]:
    return "prepare_fulfillment" if state["requested_in_stock"] else "choose_substitute"

def choose_substitute(state: OrderState) -> dict:
    candidates = SUCCESSORS[state["requested_item"]]
    index = state["candidate_index"]
    if index >= len(candidates) or state["suggestions_tried"] >= 2:
        return {"candidate_available": False, "steps": add_step(state, "No permitted substitute remains.")}
    candidate = candidates[index]
    return {"selected_item": candidate, "candidate_index": index + 1,
            "suggestions_tried": state["suggestions_tried"] + 1, "candidate_available": True,
            "steps": add_step(state, f"Suggested {candidate} (suggestion {state['suggestions_tried'] + 1} of 2).")}

def route_after_choose(state: OrderState) -> Literal["check_substitute_stock", "cancel_order"]:
    return "check_substitute_stock" if state["candidate_available"] else "cancel_order"

def check_substitute_stock(state: OrderState) -> dict:
    item = state["selected_item"]
    enough = PRODUCTS[item]["stock"] >= state["quantity"]
    return {"substitute_in_stock": enough,
            "steps": add_step(state, f"Checked suggested item {item}: {'enough stock' if enough else 'not enough stock'}.")}

def route_after_substitute_stock(state: OrderState) -> Literal["prepare_fulfillment", "choose_substitute", "cancel_order"]:
    if state["substitute_in_stock"]:
        return "prepare_fulfillment"
    candidates = SUCCESSORS[state["requested_item"]]
    has_next = state["candidate_index"] < len(candidates)
    under_limit = state["suggestions_tried"] < 2
    return "choose_substitute" if has_next and under_limit else "cancel_order"

def prepare_fulfillment(state: OrderState) -> dict:
    city = state["city"]
    if not city:
        delivery, fee, note = "pickup", 0.0, "The order is set for free shop pickup."
    elif city.casefold() == "jerusalem":
        delivery, fee, note = "delivery", 20.0, "Jerusalem delivery is 20."
    else:
        delivery, fee, note = "pickup", 0.0, f"We don't deliver to {city}, so the order is set for free pickup."
    actual = state["selected_item"]
    note += f" Your {state['quantity']} {actual} will be ready."
    if actual != state["requested_item"]:
        note = f"We're out of {state['requested_item']}, so we can offer {actual} instead. " + note
    return {"item": actual, "delivery": delivery, "delivery_fee": fee, "fulfillment_note": note,
            "steps": add_step(state, f"Prepared {delivery} fulfillment.")}

def discount_for(subtotal: float, day: str) -> float:
    return subtotal * 0.15 if subtotal > 150 and day.casefold() != "friday" else 0.0

def calculate_total(state: OrderState) -> dict:
    subtotal = PRODUCTS[state["selected_item"]]["price"] * state["quantity"]
    discount = discount_for(subtotal, state["day"])
    total = round(subtotal - discount + state["delivery_fee"], 2)
    return {"goods_subtotal": float(subtotal), "discount": float(discount), "total": total,
            "steps": add_step(state, f"Priced goods at {subtotal:.2f}; discount {discount:.2f}; delivery {state['delivery_fee']:.2f}.")}

def write_acceptance_reply(state: OrderState) -> dict:
    discount_note = f" A 15% discount saved you {state['discount']:.2f}." if state["discount"] else ""
    message = f"Thanks so much for your order! {state['fulfillment_note']}{discount_note} Your total is {state['total']:.2f}."
    return {"status": "accepted", "message": message,
            "steps": add_step(state, "Created the customer reply and completed the order.")}

def reject_order(state: OrderState) -> dict:
    return {"status": "rejected", "item": None, "delivery": "pickup", "total": 0.0,
            "message": state["invalid_reason"],
            "steps": add_step(state, "Replied politely and rejected the invalid order.")}

def cancel_order(state: OrderState) -> dict:
    item = state.get("requested_item", "order")
    return {"status": "rejected", "item": None, "delivery": "pickup", "total": 0.0,
            "message": f"I'm sorry, we're unable to fulfill {state['quantity']} {item} after checking up to two suggestions. We've canceled the order.",
            "steps": add_step(state, "Apologized and canceled after the permitted suggestions failed.")}


In [ ]:
builder = StateGraph(OrderState)
builder.add_node("validate_order", validate_order)
builder.add_node("check_requested_stock", check_requested_stock)
builder.add_node("choose_substitute", choose_substitute)
builder.add_node("check_substitute_stock", check_substitute_stock)
builder.add_node("prepare_fulfillment", prepare_fulfillment)
builder.add_node("calculate_total", calculate_total)
builder.add_node("write_acceptance_reply", write_acceptance_reply)
builder.add_node("reject_order", reject_order)
builder.add_node("cancel_order", cancel_order)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges("validate_order", route_after_validation)
builder.add_conditional_edges("check_requested_stock", route_after_requested_stock)
builder.add_conditional_edges("choose_substitute", route_after_choose)
builder.add_conditional_edges("check_substitute_stock", route_after_substitute_stock)
builder.add_edge("prepare_fulfillment", "calculate_total")
builder.add_edge("calculate_total", "write_acceptance_reply")
builder.add_edge("write_acceptance_reply", END)
builder.add_edge("reject_order", END)
builder.add_edge("cancel_order", END)
graph = builder.compile()


## 4. Graph visualization

Render the compiled LangGraph workflow as a Mermaid PNG.

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## 5. Test orders

These independent scenarios cover invalid input, delivery behavior, substitutions, cancellation, and pricing policy.

In [ ]:
test_orders = [
    ("invalid discontinued product", {"item": "croissants", "quantity": 1, "day": "Thursday", "city": ""}),
    ("invalid zero quantity", {"item": "challah", "quantity": 0, "day": "Thursday", "city": ""}),
    ("pickup", {"item": "challah", "quantity": 1, "day": "Thursday", "city": ""}),
    ("Jerusalem delivery", {"item": "sourdough", "quantity": 1, "day": "Thursday", "city": "Jerusalem"}),
    ("outside Jerusalem conversion", {"item": "challah", "quantity": 1, "day": "Thursday", "city": "Tel Aviv"}),
    ("babka to rugelach", {"item": "babka", "quantity": 2, "day": "Thursday", "city": ""}),
    ("babka through rugelach to sourdough", {"item": "babka", "quantity": 4, "day": "Thursday", "city": ""}),
    ("rugelach to sourdough", {"item": "rugelach", "quantity": 4, "day": "Thursday", "city": ""}),
    ("two failed suggestions then cancel", {"item": "babka", "quantity": 5, "day": "Thursday", "city": ""}),
    ("chain ends then cancel", {"item": "sourdough", "quantity": 11, "day": "Thursday", "city": ""}),
    ("above 150, Thursday discount", {"item": "challah", "quantity": 10, "day": "Thursday", "city": "Jerusalem"}),
    ("Friday discount excluded", {"item": "challah", "quantity": 10, "day": "Friday", "city": ""}),
]

def public_result(state: OrderState) -> dict:
    return {key: state[key] for key in PUBLIC_KEYS}

results = {}
for name, order in test_orders:
    result = graph.invoke(order)
    public = public_result(result)
    results[name] = public
    assert set(public) == PUBLIC_KEYS
    assert public["status"] in {"accepted", "rejected"}
    assert public["delivery"] in {"pickup", "delivery"}
    assert isinstance(public["steps"], list) and public["steps"]
    assert isinstance(public["total"], float) and round(public["total"], 2) == public["total"]
    print(f"{name}: {public}")


## 6. Acceptance checks

Assertions cover required behavior, totals, substitution routes, and the ordered diary.

In [ ]:
assert results["invalid discontinued product"]["status"] == "rejected"
assert results["invalid zero quantity"]["status"] == "rejected"
assert results["pickup"]["delivery"] == "pickup"
assert results["Jerusalem delivery"]["delivery"] == "delivery"
assert results["Jerusalem delivery"]["total"] == 45.0
assert results["outside Jerusalem conversion"]["delivery"] == "pickup"
assert "don't deliver" in results["outside Jerusalem conversion"]["message"]
assert results["babka to rugelach"]["item"] == "rugelach"
assert results["babka through rugelach to sourdough"]["item"] == "sourdough"
assert results["rugelach to sourdough"]["item"] == "sourdough"
assert results["two failed suggestions then cancel"]["status"] == "rejected"
assert "canceled" in results["two failed suggestions then cancel"]["message"]
assert results["chain ends then cancel"]["status"] == "rejected"
assert discount_for(150, "Thursday") == 0.0
assert round(discount_for(151, "Thursday"), 2) == 22.65
assert results["above 150, Thursday discount"]["total"] == 173.0
assert results["Friday discount excluded"]["total"] == 180.0
diary = results["babka through rugelach to sourdough"]["steps"]
assert diary.index("Suggested rugelach (suggestion 1 of 2).") < diary.index("Suggested sourdough (suggestion 2 of 2).")
assert diary[-1] == "Created the customer reply and completed the order."
print("All acceptance checks passed.")
